# 08 – LangGraph Pipeline (Full Multi-Agent Flow)

Tests the **complete end-to-end graph**: pre_hook → supervisor → agent nodes → auto_ticket → synthesizer → post_hook.  
This exercises intent classification, routing, parallel agent execution, and LLM synthesis.

> **Note**: Synthesis uses an LLM. Without `GROQ_API_KEY` set, a string fallback is used automatically.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'

In [ ]:
from graph.graph import build_graph
from graph.state import initial_state

graph = build_graph()

## 1. Simple knowledge lookup

In [ ]:
state = initial_state(query='What is the GRR threshold policy?')
result = graph.invoke(state)

print('Intent         :', result.get('intent'))
print('Agents ran     :', result.get('_agents_ran'))
print('Confidence     :', result.get('confidence'))
print('Execution ms   :', result.get('execution_ms'))
print('\nFinal summary:')
print(result.get('final_summary'))

## 2. Data quality query — metadata + information agents

In [ ]:
state = initial_state(
    query='What is the DQ score for the bookings dataset?',
    data_products=['bookings'],
)
result = graph.invoke(state)

print('Intent     :', result.get('intent'))
print('Agents ran :', result.get('_agents_ran'))
print('Sources    :', result.get('sources'))
print('\nSummary:', result.get('final_summary'))

## 3. Incident review — capacity agent

In [ ]:
state = initial_state(query='Show me open incidents and data quality issues')
result = graph.invoke(state)

print('Intent     :', result.get('intent'))
print('Agents ran :', result.get('_agents_ran'))

# Inspect agent_results
for ar in result.get('agent_results', []):
    print(f"\nAgent: {ar['agent']}  success={ar['success']}")
    data = ar.get('data') or {}
    tickets = data.get('tickets', [])
    if tickets:
        print(f'  {len(tickets)} tickets found')
        for t in tickets[:3]:
            print(f"  [{t['id']}] {t['summary']} ({t['status']})")

## 4. Full diagnostic — all four read agents

In [ ]:
state = initial_state(
    query='Give me a full diagnostic overview of all data products',
    data_products=['retention', 'bookings', 'cac', 'ltv'],
)
result = graph.invoke(state)

print('Intent      :', result.get('intent'))
print('Agents ran  :', result.get('_agents_ran'))
print('Anomalies   :', result.get('anomalies'))
print('Errors      :', result.get('errors'))
print('Confidence  :', result.get('confidence'))
print('Exec ms     :', result.get('execution_ms'))
print('\nFinal summary:\n', result.get('final_summary'))

## 5. Write rule intent

In [ ]:
state = initial_state(query='Create a new data quality rule for retention completeness')
result = graph.invoke(state)

print('Intent     :', result.get('intent'))
print('Agents ran :', result.get('_agents_ran'))

for ar in result.get('agent_results', []):
    if ar['agent'] == 'rule':
        data = ar.get('data') or {}
        if isinstance(data, dict) and 'id' in data:
            print(f"Rule created: {data['id']} — {data.get('name')}")

## 6. State inspection — all fields

In [ ]:
state = initial_state(query='What are the governance policies for LTV?')
result = graph.invoke(state)

print('State keys after pipeline:')
for k, v in result.items():
    if k.startswith('_') or k in ('conversation_history', 'user_preferences'):
        continue
    v_repr = str(v)[:80] if not isinstance(v, (list, dict)) else f'({type(v).__name__}, len={len(v)})'
    print(f'  {k:<25} {v_repr}')